In [ ]:
# Run lsoa.ipynb first to generate the LSOA polygons and seed nodes

In [ ]:
from random import Random

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from shapely import wkt
from sklearn.cluster import MiniBatchKMeans

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_GEO, CRS_METRIC, REGION, SEED

In [ ]:
ox.settings.cache_folder = paths.preprocessing.cache

In [ ]:
prepare_preprocessing()

In [ ]:
GRAPH_OUT_PATH = paths.preprocessing.shared.network
GENERATE_MISSING_DATA = not GRAPH_OUT_PATH.exists()

In [ ]:
# Retrieve the LSOA boundaries from the MEDSAT dataset
medsat = pd.read_csv(paths.raw.shared.medsat)
medsat['geometry'] = medsat['geometry'].apply(wkt.loads)
medsat = medsat[['geography code', 'LSOA21NM', 'geometry']].rename(columns = {'geography code': 'LSOA21CD'})

polygons = gpd.GeoDataFrame(medsat, geometry = 'geometry', crs = CRS_METRIC)

In [ ]:
# Clip the LSOA polygons to the study area
region = ox.geocode_to_gdf(REGION).to_crs(CRS_METRIC)
region_buffered = region.buffer(2000)

polygons['centroid'] = polygons.geometry.centroid
mask = polygons['centroid'].within(region.union_all())

lsoa = polygons[mask].copy()

In [ ]:
# If the walkable network does not exist, generate it, else load it
if GENERATE_MISSING_DATA:
    custom_filter = (
        '["highway"]'
        '["area"!~"yes"]'
        '["highway"!~"abandoned|bus_guideway|construction|motor|no|planned|platform|proposed|raceway|razed"]'
        '["foot"!~"no"]'
        '["service"!~"private"]'
        '["sidewalk"!~"separate"]'
        '["sidewalk:both"!~"separate"]'
        '["sidewalk:left"!~"separate"]'
        '["sidewalk:right"!~"separate"]'
    )

    G = ox.graph_from_polygon(
        region_buffered.to_crs(CRS_GEO).geometry.union_all(),
        simplify = True,
        retain_all = True,
        truncate_by_edge = False,
        custom_filter = custom_filter,
    )

    allowed = {'permissive', 'yes', 'designated', 'destination'}
    to_keep = [
        (u, v, k)
        for u, v, k, d in G.edges(keys = True, data = True)
        if 'access' not in d
        or (isinstance(d['access'], list) and any(a in allowed for a in d['access']))
        or (isinstance(d['access'], str) and d['access'] in allowed)
    ]

    G = G.edge_subgraph(to_keep).copy()

    wccs = list(nx.weakly_connected_components(G))
    nodes_to_remove = set()

    for component in wccs:
        if len(component) < 500:
            nodes_to_remove.update(component)

    G.remove_nodes_from(nodes_to_remove)
    G = ox.project_graph(G, to_crs = CRS_METRIC)
    G = nx.MultiGraph(G)

    ox.save_graphml(G, GRAPH_OUT_PATH)

else:
    G = ox.load_graphml(GRAPH_OUT_PATH)

In [ ]:
# Select the seed nodes for each LSOA and add them to the LSOA GDF
nodes, edges = ox.graph_to_gdfs(G)
def pick_seeds(group, k, random_state = SEED):
    node_ids = group.index.to_numpy()
    n = len(node_ids)

    if n == k:
        return list(node_ids)

    coords = np.c_[group.geometry.x.values, group.geometry.y.values]
    km = MiniBatchKMeans(n_clusters = k, random_state = random_state).fit(coords)
    labels = km.labels_
    centres = km.cluster_centers_

    seeds = []

    for i in range(k):
        members = np.where(labels == i)[0]

        if members.size:
            centre = centres[i]
            nearest = members[np.argmin(((coords[members] - centre) ** 2).sum(1))]
            seeds.append(node_ids[nearest])

    if len(seeds) < k:
        remaining_nodes = sorted(set(node_ids) - set(seeds))
        extra = Random(random_state).sample(remaining_nodes, k - len(seeds))
        seeds.extend(extra)

    return seeds

def seeds_for_lsoa(lsoa, k = 10, m = 20):
    geom = lsoa.geometry

    mask = nodes.within(geom)
    group = nodes.loc[mask]

    while len(group) < k:
        geom = geom.buffer(m)
        group = nodes.loc[nodes.within(geom)]

    return pick_seeds(group, k, random_state = SEED)

seed_cols = [f'sn{i}' for i in range(10)]
seed_df = lsoa.apply(seeds_for_lsoa, axis = 1, result_type = 'expand')
seed_df.columns = seed_cols

lsoa_with_seeds = pd.concat([lsoa, seed_df], axis = 1)

In [ ]:
lsoa_with_seeds.to_parquet(paths.preprocessing.shared.lsoa)